In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import xarray as xr

basedir = "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_region_variables/"
outdir  = "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/"

# ----------------------------
# Helper: load -> flatten -> drop NaNs
# ----------------------------
def load_flat_clean(nc_path, varname):
    ds = xr.open_dataset(nc_path)
    arr = ds[varname].values.flatten()
    arr = arr[~np.isnan(arr)]
    ds.close()
    return arr

# ----------------------------
# Normalize bin centers to 0–1
# ----------------------------
def normalize_x(x, lognorm=True):
    x = np.asarray(x)
    if lognorm:
        x = x[x > 0]
        lx = np.log10(x)
        return (lx - lx.min()) / (lx.max() - lx.min())
    else:
        return (x - x.min()) / (x.max() - x.min())

# ----------------------------
# Compute PDF (%) on given bins
# ----------------------------
def hist_percent_on_bins(arr, bins):
    counts, _ = np.histogram(arr, bins=bins)
    y = counts / counts.sum() * 100.0
    centers = 0.5 * (bins[:-1] + bins[1:])
    return centers, y

# ----------------------------
# File maps
# ----------------------------
files_wp = {
    "CWP": {
        "core":       basedir + "Obs_cwp_core_mcs_20160809-20160909_Asia_timeavg.nc",
        "cold anvil": basedir + "Obs_cwp_cold_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
        "warm anvil": basedir + "Obs_cwp_warm_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
    },
    "CIW": {
        "core":       basedir + "Obs_ciw_core_mcs_20160809-20160909_Asia_timeavg.nc",
        "cold anvil": basedir + "Obs_ciw_cold_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
        "warm anvil": basedir + "Obs_ciw_warm_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
    },
    "CLW": {
        "core":       basedir + "Obs_clw_core_mcs_20160809-20160909_Asia_timeavg.nc",
        "cold anvil": basedir + "Obs_clw_cold_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
        "warm anvil": basedir + "Obs_clw_warm_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
    }
}

files_pr = {
    "CWP": {
        "core":       basedir + "Obs_pr_cwp_core_mcs_20160809-20160909_Asia_timeavg.nc",
        "cold anvil": basedir + "Obs_pr_cwp_cold_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
        "warm anvil": basedir + "Obs_pr_cwp_warm_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
    },
    "CIW": {
        "core":       basedir + "Obs_pr_ciw_core_mcs_20160809-20160909_Asia_timeavg.nc",
        "cold anvil": basedir + "Obs_pr_ciw_cold_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
        "warm anvil": basedir + "Obs_pr_ciw_warm_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
    },
    "CLW": {
        "core":       basedir + "Obs_pr_clw_core_mcs_20160809-20160909_Asia_timeavg.nc",
        "cold anvil": basedir + "Obs_pr_clw_cold_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
        "warm anvil": basedir + "Obs_pr_clw_warm_anvil_mcs_20160809-20160909_Asia_timeavg.nc",
    }
}

# ----------------------------
# Load data
# ----------------------------
regions = ["core", "cold anvil", "warm anvil"]
vars_order = ["CWP", "CIW", "CLW"]

wp = {v: {r: load_flat_clean(files_wp[v][r], v) for r in regions} for v in vars_order}
pr = {v: {r: load_flat_clean(files_pr[v][r], "precipitation") for r in regions} for v in vars_order}

# ----------------------------
# Plot settings
# ----------------------------
colors = {"core": "blue", "cold anvil": "red", "warm anvil": "green"}

legend_handles = [
    Line2D([0], [0], color=colors["core"], lw=3, label="core"),
    Line2D([0], [0], color=colors["cold anvil"], lw=3, label="cold anvil"),
    Line2D([0], [0], color=colors["warm anvil"], lw=3, label="warm anvil"),
    Line2D([0], [0], color="k", lw=2.5, ls="--", label="precipitation"),
]

xlabel_map = {
    "CWP": r"$\epsilon_\mathrm{cwp}$",
    "CIW": r"$\epsilon_i$",
    "CLW": r"$\epsilon_\ell$",
}

LOGNORM_X = True
NBINS = 60

def make_positive_bins_from_data(arr_list, nbins=60, logbins=True):
    allv = np.concatenate(arr_list)
    allv = allv[np.isfinite(allv)]
    if logbins:
        allv = allv[allv > 0]
        vmin = np.nanpercentile(allv, 1)
        vmax = np.nanpercentile(allv, 99.5)
        bins = np.logspace(np.log10(vmin), np.log10(vmax), nbins)
    else:
        vmin = np.nanpercentile(allv, 1)
        vmax = np.nanpercentile(allv, 99.5)
        bins = np.linspace(vmin, vmax, nbins)
    return bins

# ----------------------------
# Create 3x1 subplots
# ----------------------------
fig, axes = plt.subplots(3, 1, figsize=(10, 16), sharex=True)
panel_labels = ["(a)", "(b)", "(c)"]

for i, var in enumerate(vars_order):
    ax = axes[i]

    bins_wp = make_positive_bins_from_data([wp[var][r] for r in regions], NBINS, LOGNORM_X)
    bins_pr = make_positive_bins_from_data([pr[var][r] for r in regions], NBINS, LOGNORM_X)

    centers_wp = 0.5 * (bins_wp[:-1] + bins_wp[1:])
    centers_pr = 0.5 * (bins_pr[:-1] + bins_pr[1:])

    x_wp = normalize_x(centers_wp, LOGNORM_X)
    x_pr = normalize_x(centers_pr, LOGNORM_X)

    for region in regions:
        _, y_wp = hist_percent_on_bins(wp[var][region], bins_wp)
        _, y_pr = hist_percent_on_bins(pr[var][region], bins_pr)

        ax.plot(x_wp, y_wp, color=colors[region], lw=3)
        ax.plot(x_pr, y_pr, color=colors[region], lw=2.5, ls="--")

    # ---- Formatting ----
    ax.set_ylabel("Probability [%]", fontsize=25)
    ax.tick_params(axis="both", labelsize=25)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.yaxis.grid(True, linestyle=":", linewidth=1)
    ax.xaxis.grid(False)

    ax.set_xlabel(xlabel_map[var], fontsize=30, labelpad=12)

    ax.text(0.90, 0.90, rf"$\bf{{{panel_labels[i]}}}$",
            transform=ax.transAxes, fontsize=30)

# Legend on top panel
axes[0].legend(handles=legend_handles, fontsize=20, loc="upper left", frameon=False)

axes[-1].set_xlim(0, 1)
axes[0].set_xlabel(r"Normalized Cloud Water Path", fontsize=25)
axes[1].set_xlabel(r"Normalized Ice Water Path", fontsize=25)
axes[2].set_xlabel(r"Normalized Liquid Water Path", fontsize=25)

plt.tight_layout()
plt.savefig(outdir + "FigureS4_Obs_cloud_condensate_vs_precip_histograms_3x1_normx.png",
            dpi=150, bbox_inches="tight")
plt.savefig(outdir + "FigureS4_Obs_cloud_condensate_vs_precip_histograms_3x1_normx.pdf",
            dpi=150, bbox_inches="tight")
#plt.show()
